#### CREDIT PORTFOLIO HEALTH MONITOR
- Data Generation

#### Imports

In [3]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)

fake = Faker('en_KE')

print("Environment Ready for MoPhones Credit Portfolio Data Generation")
print(f"Current Working Directory: {os.getcwd()}")

Environment Ready for MoPhones Credit Portfolio Data Generation
Current Working Directory: C:\Users\tohiba\Desktop\credit_portfolio_health_monitor\notebooks


#### Business Parameters

In [4]:
NUM_CUSTOMERS = 15_000
START_DATE = '2023-01-01'
END_DATE = '2025-03-31'

print(f"Generating data for {NUM_CUSTOMERS:,} customers from {START_DATE} to {END_DATE}")

Generating data for 15,000 customers from 2023-01-01 to 2025-03-31


#### Generate Customers Table

In [5]:
customers = pd.DataFrame({
    'customer_id': [f'CUST{str(i).zfill(6)}' for i in range(1, NUM_CUSTOMERS + 1)],
    'full_name': [fake.name() for _ in range(NUM_CUSTOMERS)],
    'phone_number': [fake.phone_number() for _ in range(NUM_CUSTOMERS)],
    'email': [fake.email() for _ in range(NUM_CUSTOMERS)],
    'age': np.random.randint(21, 55, NUM_CUSTOMERS),
    'gender': np.random.choice(['Male', 'Female'], NUM_CUSTOMERS, p=[0.62, 0.38]),
    'location': np.random.choice(['Nairobi', 'Mombasa', 'Kisumu', 'Eldoret', 'Nakuru', 'Other'], 
                                NUM_CUSTOMERS, p=[0.45, 0.15, 0.10, 0.08, 0.12, 0.10]),
    'income_band': np.random.choice(['Low', 'Lower_Mid', 'Middle', 'Upper_Mid'], 
                                   NUM_CUSTOMERS, p=[0.35, 0.40, 0.20, 0.05]),
    'credit_score': np.random.randint(300, 850, NUM_CUSTOMERS),
    'created_at': np.random.choice(pd.date_range(START_DATE, END_DATE), NUM_CUSTOMERS)
})

customers['created_at'] = pd.to_datetime(customers['created_at'])

print("Customers Table Generated")
print(f"Shape: {customers.shape}")
customers.head()

Customers Table Generated
Shape: (15000, 10)


,customer_id,full_name,phone_number,email,age,gender,location,income_band,credit_score,created_at
0,CUST000001,Seth Kariuki,9979690844,mercynduku@example.com,49,Male,Eldoret,Lower_Mid,366,2023-08-12
1,CUST000002,Bishop Elisha Godana,511.863.3798x221,njokiann@example.org,35,Male,Other,Low,814,2024-08-01
2,CUST000003,Dr Mary Chomba,(810)963-5717,mathew70@example.com,28,Male,Nairobi,Middle,454,2023-12-26
3,CUST000004,Ruth Wangari,833.896.7642x982,hsoita@example.net,41,Female,Mombasa,Lower_Mid,812,2024-07-07
4,CUST000005,Ms Leah Mohamed,+1-464-666-2818x421,dnthiga@example.com,39,Female,Nakuru,Lower_Mid,634,2025-01-24


#### Generate Loans Table

In [6]:
loans = []
loan_id_counter = 1

date_range = pd.date_range(START_DATE, END_DATE)

for _, cust in customers.iterrows():
    num_loans = np.random.poisson(1.8)  
    
    for _ in range(int(num_loans)):
        disbursed_date = cust['created_at'] + timedelta(days=random.randint(0, 450))
        if disbursed_date > pd.to_datetime(END_DATE):
            continue
            
        loan_amount = round(np.random.lognormal(8.5, 0.6), -2)  
        term_months = random.choice([3, 6, 9, 12])
        interest_rate = round(random.uniform(1.8, 4.5), 2)
        
        loans.append({
            'loan_id': f'LOAN{str(loan_id_counter).zfill(6)}',
            'customer_id': cust['customer_id'],
            'disbursed_date': disbursed_date.date(),
            'principal': loan_amount,
            'term_months': term_months,
            'interest_rate': interest_rate,
            'device_model': random.choice(['iPhone 11', 'Samsung Galaxy A15', 'Redmi 13C', 
                                         'TECNO Spark 20', 'iPhone 12', 'Samsung A35', 'Infinix Hot 40']),
            'promo_applied': random.choice([True, False, False, False])
        })
        loan_id_counter += 1

loans_df = pd.DataFrame(loans)

print("Loans Table Generated")
print(f"Total Loans: {len(loans_df):,}")
loans_df.head()

Loans Table Generated
Total Loans: 19,692


,loan_id,customer_id,disbursed_date,principal,term_months,interest_rate,device_model,promo_applied
0,LOAN000001,CUST000001,2024-07-04,4100.0,3,1.87,Redmi 13C,False
1,LOAN000002,CUST000003,2024-04-18,6500.0,6,3.79,Samsung A35,True
2,LOAN000003,CUST000003,2024-10-23,8400.0,12,1.89,iPhone 11,False
3,LOAN000004,CUST000003,2024-04-23,3300.0,3,3.32,Samsung A35,False
4,LOAN000005,CUST000003,2024-04-16,3800.0,12,3.39,Infinix Hot 40,True


#### Generate Repayments Table

In [7]:
repayments = []

for _, loan in loans_df.iterrows():
    monthly_payment = round(loan['principal'] * 1.28 / loan['term_months'], 2)
    num_payments = int(loan['term_months'] * 1.15) 
    
    for i in range(num_payments):
        due_date = loan['disbursed_date'] + timedelta(days=30 * (i + 1))
        if due_date > pd.to_datetime(END_DATE).date():
            break
            
        days_late = random.choices([0, 3, 8, 18, 35, 60], weights=[0.72, 0.10, 0.08, 0.05, 0.03, 0.02])[0]
        
        paid_date = due_date + timedelta(days=days_late) if days_late > 0 else due_date
        
        if paid_date > pd.to_datetime(END_DATE).date():
            amount_paid = 0.0
            paid_date = None
        else:
            amount_paid = round(monthly_payment * random.uniform(0.65, 1.0), 2)
        
        repayments.append({
            'repayment_id': f'REP{str(len(repayments)+1).zfill(7)}',
            'loan_id': loan['loan_id'],
            'due_date': due_date,
            'paid_date': paid_date,
            'amount_due': monthly_payment,
            'amount_paid': amount_paid,
            'days_late': days_late
        })

repayments_df = pd.DataFrame(repayments)

print("Repayments Table Generated")
print(f"Total Repayment Records: {len(repayments_df):,}")
repayments_df.head()

Repayments Table Generated
Total Repayment Records: 115,072


,repayment_id,loan_id,due_date,paid_date,amount_due,amount_paid,days_late
0,REP0000001,LOAN000001,2024-08-03,2024-08-06,1749.33,1301.15,3
1,REP0000002,LOAN000001,2024-09-02,2024-09-02,1749.33,1497.64,0
2,REP0000003,LOAN000001,2024-10-02,2024-10-02,1749.33,1644.77,0
3,REP0000004,LOAN000002,2024-05-18,2024-05-18,1386.67,1229.91,0
4,REP0000005,LOAN000002,2024-06-17,2024-06-17,1386.67,1195.26,0


#### Quick Validation

In [8]:
print("=== DATA VALIDATION SUMMARY ===")
print(f"Total Customers : {len(customers):,}")
print(f"Total Loans     : {len(loans_df):,}")
print(f"Total Repayments: {len(repayments_df):,}")
print(f"Avg Loans per Customer: {len(loans_df)/len(customers):.2f}")

# Portfolio Value
print(f"Total Portfolio Disbursed (KES): {loans_df['principal'].sum():,.0f}")

=== DATA VALIDATION SUMMARY ===
Total Customers : 15,000
Total Loans     : 19,692
Total Repayments: 115,072
Avg Loans per Customer: 1.31
Total Portfolio Disbursed (KES): 115,201,900


#### Save Raw Data



In [11]:
os.makedirs('../data/raw', exist_ok=True)

customers.to_csv('../data/raw/customers.csv', index=False)
loans_df.to_csv('../data/raw/loans.csv', index=False)
repayments_df.to_csv('../data/raw/repayments.csv', index=False)

print("All raw data successfully saved!")
print("\nFiles Created at: ../data/raw/")
print("- customers.csv")
print("- loans.csv")
print("- repayments.csv")

All raw data successfully saved!

Files Created at: ../data/raw/
- customers.csv
- loans.csv
- repayments.csv
